In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

torch.backends.cudnn.benchmark = True 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge"
train_df = pd.read_csv(f"{BASE_DIR}/train_metadata.csv")

L = train_df['label'].unique()
dict_labels = {L[i]: i for i in range(len(L))}
inv_dict_labels = {v: k for k, v in dict_labels.items()}

y_train_full = train_df["label"].map(dict_labels).astype(int).values
train_df['target'] = y_train_full

train_transforms = transforms.Compose([
    transforms.Resize((300, 300)), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BloodCellDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.df.iloc[idx]['ID'])
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, torch.tensor(self.df.iloc[idx]['target'], dtype=torch.long)

In [ ]:
# PARAMETERS TO CHANGE
TARGET_FOLD = 4 # The "Golden Fold"
model_name = "resnet"
# Other models to select : "densenet", "efficientnet_b3", "efficientnet_v2", "convnext"
# ==========================================

epochs = 35
batch_size = 16

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = torch.amp.GradScaler('cuda')

print(f"Training {model_name} on fold {TARGET_FOLD}")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y_train_full)):
    current_fold = fold + 1
    
    # Skip all folds except the one assigned to this specific notebook!
    if current_fold != TARGET_FOLD:
        continue
    
    train_data = BloodCellDataset(train_df.iloc[train_idx], f"{BASE_DIR}/train", transform=train_transforms)
    val_data = BloodCellDataset(train_df.iloc[val_idx], f"{BASE_DIR}/train", transform=val_test_transforms)
    
    fold_train_targets = y_train_full[train_idx]
    counts = np.bincount(fold_train_targets)
    weights = 1.0 / np.sqrt(counts)
    sample_weights = weights[fold_train_targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_data, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    if model_name=="efficientnet_b3":
        model = models.efficientnet_b3(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(L))
        
    elif model_name=="resnet":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(L))
        
    elif model_name=="convnet":
        model = models.convnext_tiny(weights=None)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(L))
        
    elif model_name=="efficientnet_v2":
        model = models.efficientnet_v2_s(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(L))
        
    elif model_name=="densenet":
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, len(L))
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # label smoothing mentionned on the report
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_f1 = 0.0
    best_weights_path = f"Fold_{current_fold}_{model_name}.pth"
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                _, p = torch.max(outputs, 1)
                preds.extend(p.cpu().numpy())
                targets.extend(labels.cpu().numpy())
                
        ep_f1 = f1_score(targets, preds, average='macro')
        print(f"Fold {current_fold} | Ep [{epoch+1}/{epochs}] | Val F1: {ep_f1:.4f}")
        
        if ep_f1 > best_f1:
            # Saving the best model so far
            best_f1 = ep_f1
            torch.save(model.state_dict(), best_weights_path)
            
        scheduler.step()
        
    print(f"Fold {current_fold} Complete! Best F1: {best_f1:.4f}. Weights saved to output.")
    break 